# Frenet-Frame Planner Demo

This notebook is a self-learning walkthrough of a simplified Frenet-frame trajectory generator.

You will sample lateral-offset candidates in Frenet coordinates, convert them to Cartesian space, and choose the best feasible trajectory.

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np

## 1. Build a reference path and environment

In [ ]:
s_ref = np.linspace(0.0, 55.0, 350)
x_ref = s_ref
y_ref = 2.0 * np.sin(0.12 * s_ref)

obstacles = np.array([
    [18.0, 1.7], [24.0, 0.9], [30.0, -0.2], [35.0, -1.0], [42.0, -1.5]
])

s_start = 5.0
d_start = 0.0

## 2. Frenet helper functions

Key mapping idea:
- `s` moves along the reference path.
- `d` offsets laterally using the local normal direction.

In [ ]:
def interp_ref(sq):
    x = np.interp(sq, s_ref, x_ref)
    y = np.interp(sq, s_ref, y_ref)
    return x, y

def tangent_normal(sq):
    ds = 0.15
    s0 = np.clip(sq - ds, s_ref[0], s_ref[-1])
    s1 = np.clip(sq + ds, s_ref[0], s_ref[-1])
    x0, y0 = interp_ref(s0)
    x1, y1 = interp_ref(s1)

    tx, ty = x1 - x0, y1 - y0
    nrm = np.hypot(tx, ty) + 1e-9
    tx, ty = tx / nrm, ty / nrm

    # Left normal
    nx, ny = -ty, tx
    return tx, ty, nx, ny

def frenet_to_xy(s_seq, d_seq):
    xs, ys = [], []
    for ss, dd in zip(s_seq, d_seq):
        xr, yr = interp_ref(ss)
        _, _, nx, ny = tangent_normal(ss)
        xs.append(xr + dd * nx)
        ys.append(yr + dd * ny)
    return np.array(xs), np.array(ys)

## 3. Sample candidate lateral profiles

We sample final lateral offsets and interpolate from current `d_start` to each candidate target.

In [ ]:
horizon_s = 22.0
num_pts = 80
s_path = np.linspace(s_start, s_start + horizon_s, num_pts)

d_targets = np.linspace(-1.6, 1.6, 11)

candidates = []
for dT in d_targets:
    u = np.linspace(0.0, 1.0, num_pts)
    # Cubic blend with zero slope at both ends
    blend = 3 * u**2 - 2 * u**3
    d_path = d_start + (dT - d_start) * blend

    x_path, y_path = frenet_to_xy(s_path, d_path)
    candidates.append({
        'd_target': dT,
        's': s_path.copy(),
        'd': d_path,
        'x': x_path,
        'y': y_path
    })

len(candidates)

## 4. Score and select the best candidate

Cost terms in this simplified demo:
- lateral effort/smoothness,
- obstacle clearance penalty,
- preference for smaller final offset magnitude.

In [ ]:
w_lat = 1.1
w_obs = 2.6
w_center = 0.35
safe_dist = 1.0

def candidate_cost(c):
    d = c['d']
    x, y = c['x'], c['y']

    # Lateral smoothness effort
    lat_cost = np.mean(np.diff(d, n=2) ** 2)

    # Obstacle penalty
    pts = np.column_stack([x, y])
    obs_cost = 0.0
    for p in pts:
        dmin = np.min(np.linalg.norm(obstacles - p, axis=1))
        if dmin < safe_dist:
            obs_cost += (safe_dist - dmin) ** 2

    center_cost = abs(c['d_target'])

    total = w_lat * lat_cost + w_obs * obs_cost + w_center * center_cost
    return total, lat_cost, obs_cost, center_cost

scored = []
for c in candidates:
    total, lat_c, obs_c, ctr_c = candidate_cost(c)
    rec = dict(c)
    rec.update({'total': total, 'lat': lat_c, 'obs': obs_c, 'ctr': ctr_c})
    scored.append(rec)

best = min(scored, key=lambda r: r['total'])
print('Best d_target:', round(best['d_target'], 3))
print('Best total cost:', round(best['total'], 4))

## 5. Visualize candidates and selected trajectory

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(x_ref, y_ref, color='gray', linestyle='--', linewidth=1.4, label='reference path')
ax.scatter(obstacles[:, 0], obstacles[:, 1], c='black', s=35, label='obstacles')

for r in scored:
    ax.plot(r['x'], r['y'], color='lightsteelblue', alpha=0.5, linewidth=1.0)

ax.plot(best['x'], best['y'], color='deepskyblue', linewidth=2.6, label='selected trajectory')

x0, y0 = frenet_to_xy(np.array([s_start]), np.array([d_start]))
ax.scatter(x0[0], y0[0], c='limegreen', s=80, marker='o', label='current state')
ax.scatter(best['x'][-1], best['y'][-1], c='crimson', s=80, marker='*', label='horizon end')

ax.set_title('Frenet candidate trajectories and selected path')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_aspect('equal')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

## 6. Self-study experiments

Try these experiments:
1. Increase `horizon_s` and observe whether farther lookahead changes chosen offset.
2. Reduce `safe_dist` and compare obstacle clearance behavior.
3. Add more obstacles near the reference centerline and inspect lane-like offset selection.